# Powering the Planet: End-to-End Walkthrough
**Course:** Coding for Data Science and Data Management (Università degli Studi di Milano)

This notebook demonstrates how the `powering_planet` backend package decouples data ingestion, domain modeling (`CountryProfile`, `EnergyDataset`), statistical analysis, and Plotly visualizations from the Streamlit web interface.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from powering_planet import EnergyDataset
from powering_planet.analysis import gdp_co2_correlation, simulate_transition
from powering_planet.metrics import ENERGY_GROUPS
from powering_planet.plots import (
    plot_decoupling_index,
    plot_kuznets_scatter,
    plot_stacked_energy,
)

ds = EnergyDataset.load(ROOT / "data" / "dataset.csv")
print(f"Loaded dataset: {ds.df.shape[0]:,} rows across {ds.df['country'].nunique()} countries.")

Loaded dataset: 11,700 rows across 220 countries.


## 1. Country Profile & Economic-Emissions Decoupling
Using `CountryProfile`, we examine Italy's primary energy mix evolution (1990–2023) and test whether economic growth (GDP per capita PPP) decoupled from $\text{CO}_2$ emissions.

In [2]:
italy = ds.get_country("Italy", year_range=(1990, 2023))

decoupling = italy.decoupling_summary()
print("Italy Decoupling Summary (1990–2023):")
for key, val in decoupling.items():
    print(f"  - {key}: {val:.2f}%" if isinstance(val, float) else f"  - {key}: {val}")

fig_mix = plot_stacked_energy(
    italy.df,
    ENERGY_GROUPS["low_carbon"],
    total_col="low_carbon_energy_share_pct",
)
fig_mix.show()

fig_decoupling = plot_decoupling_index(italy.df)
fig_decoupling.show()

Italy Decoupling Summary (1990–2023):
  - status: Absolute Decoupling
  - gdp_cagr: 0.77%
  - co2_cagr: -1.05%


## 2. Global Energy Transition Leaderboard
`EnergyDataset.ranking()` filters cross-sectional country snapshots for a target year, ranking economies by low-carbon primary energy share alongside fossil reliance and carbon intensity.

In [3]:
top_2022 = ds.ranking(2022)
cols = [
    "rank",
    "country",
    "low_carbon_energy_share_pct",
    "fossil_fuel_share_pct",
    "co2_per_capita",
    "gdp_pc_ppp",
]
display(top_2022[[c for c in cols if c in top_2022.columns]].head(10))

,rank,country,low_carbon_energy_share_pct,co2_per_capita,gdp_pc_ppp
0,1,Sweden,66.398260,3.563982,62544.500
1,2,Finland,50.651800,6.523773,57068.305
2,3,Norway,48.476463,7.476182,95109.820
3,4,New Zealand,46.718620,6.213688,49842.984
4,5,France,46.000088,4.448486,53673.812
5,6,Switzerland,43.409330,3.749430,85762.280
6,7,Brazil,34.616222,2.285464,18554.047
7,8,Slovenia,33.412724,6.015118,47058.887
8,9,Slovakia,32.604656,5.763820,38648.984
9,10,Ukraine,31.746107,3.484633,13787.121


## 3. Cross-Sectional GDP–$\text{CO}_2$ Relationship & Policy Simulation
Finally, we evaluate the cross-country statistical relationship between economic output and per-capita emissions (`gdp_co2_correlation`) and run a counterfactual fossil-to-low-carbon substitution scenario (`simulate_transition`).

In [5]:
corr_stats = gdp_co2_correlation(ds.df, year=2022)
print("2022 Log-Log GDP vs. CO2 Correlation:")
for k, v in corr_stats.items():
    print(f"  - {k}: {v:.4f}" if isinstance(v, float) else f"  - {k}: {v}")

fig_scatter = plot_kuznets_scatter(ds.df, year=2022)
fig_scatter.show()

2022 Log-Log GDP vs. CO2 Correlation:
  - n: 190
  - spearman_r: 0.8849
  - spearman_p: 0.0000
  - pearson_r: 0.6552


In [7]:
germany_2022 = ds.get_country("Germany", year_range=(2022, 2022)).df.iloc[0]

sim_result = simulate_transition(
    germany_2022,
    coal_cut_pct=50.0,
    gas_cut_pct=25.0,
)

print("Counterfactual Simulation (Germany 2022: -50% Coal, -25% Gas):")
for k, v in sim_result.items():
    unit = "%" if k == "reduction_pct" else " Mt"
    print(f"  - {k}: {v:,.2f}{unit}")

Counterfactual Simulation (Germany 2022: -50% Coal, -25% Gas):
  - baseline_co2_mt: 670.02 Mt
  - saved_co2_mt: 142.17 Mt
  - new_co2_mt: 527.85 Mt
  - reduction_pct: 21.22%
